In [1]:
from my_feature_builder import build_features
from defenders.pii_detection.crf.model import LinearCRF
import pandas as pd
from transformers import AutoTokenizer
from defenders.pii_detection.src.utils import prepare_dataset
from sklearn.metrics import classification_report,accuracy_score

In [2]:
path_to_data = "./../data_pii/data.parquet"
manager = build_features()
df = pd.read_parquet(path_to_data)
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")
df=prepare_dataset(df, tokenizer)
df["tokens"] = df["words"]
df["labels"] = df["labels"]
df = df[["tokens", "labels"]]
df=df.sample(n=12, random_state=42).reset_index(drop=True)

In [3]:
dataset = df.values.tolist()
labels = set()

for _, y in dataset:
    labels.update(y)

labels = sorted(labels)

In [5]:
model = LinearCRF(feature_manager=manager,labels=labels,lr=0.05,epochs=5,l2=1e-4)
model.fit(dataset)

Epoch 1/5 ,Training Accuracy:0.976990861618799


KeyboardInterrupt: 

In [ ]:
model.save_model("crf_from_Scratch_weights2.json")

In [6]:
prediction = model.predict(["my","email","john@gmail.com"])
print(prediction)

['O', 'O', 'B-EMAIL']


In [9]:
path_to_data = "./../data_pii/test.parquet"
manager = build_features()
df_test = pd.read_parquet(path_to_data)
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")
df_test=prepare_dataset(df_test, tokenizer)
df_test["tokens"] = df_test["words"]
df_test["labels"] = df_test["labels"]
df_test = df_test[["tokens", "labels"]]

In [18]:
def evaluate(model, df_test):
    all_predictions = []
    true_labels = []

    for i in range(len(df_test)):
        text = df_test["tokens"].iloc[i]
        predictions = model.predict(text)
        predicted_labels = [label[2:] for label in predictions]
        gold_labels = [label[2:] for label in df_test["labels"].iloc[i]]
        if len(predicted_labels) != len(gold_labels):
            continue

        all_predictions.extend(predicted_labels)
        true_labels.extend(gold_labels)

    accuracy = accuracy_score(true_labels, all_predictions)
    print(f"accuracy: {accuracy}")
    print(classification_report(true_labels, all_predictions))

In [19]:
evaluate(model, df_test)

accuracy: 0.9778956540348839
                  precision    recall  f1-score   support

                       0.99      0.99      0.99     63885
     ACCOUNTNAME       0.00      0.00      0.00       281
   ACCOUNTNUMBER       0.00      0.00      0.00       110
CREDITCARDNUMBER       0.23      0.64      0.34        89
           EMAIL       0.91      1.00      0.95       153
            IPV4       0.53      0.80      0.64       120
            IPV6       0.03      0.02      0.02       105
             MAC       0.92      1.00      0.96        77
        PASSWORD       0.58      0.39      0.46       101
    PHONE_NUMBER       0.00      0.00      0.00       214
             SSN       0.33      0.01      0.01       137
        USERNAME       0.53      0.23      0.32       145

        accuracy                           0.98     65417
       macro avg       0.42      0.42      0.39     65417
    weighted avg       0.97      0.98      0.97     65417



d:\anaconda\envs\genai-env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\anaconda\envs\genai-env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\anaconda\envs\genai-env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
